# Lab 1: Privacy Policy Analysis Using NLP

![Privacy Policy](https://arc-anglerfish-washpost-prod-washpost.s3.amazonaws.com/public/Z6FEFNRRA5AOZIHGIVXUJ76YNU.jpg)

Privacy policies are often long and difficult to understand.

Most users do not read them, even though they contain important information about how data is collected and used.

## Understanding How Apps Use Your Data

Every time you install an app, it collects and uses your data.

However, this information is often hidden inside long and complex privacy policies that most users do not read.

---

## Real World Example

Consider the following two sentences:

- We collect your email address and share it with advertisers  
- This policy may be updated from time to time  

### Question

Which sentence actually tells you how your data is used?

### Answer

The first sentence clearly explains how your data is collected and shared.  
The second sentence does not.

## Goal of This Lab

Build a system that can automatically detect sentences that describe how user data is handled.

These sentences are called **data practice sentences**.

---

## What You Will Learn

In this lab, you will learn how to:

- Understand what a **data practice sentence** is  
- Work with a **real-world privacy policy dataset**  
- Apply simple **text processing techniques**  
- Build a basic **keyword-based detection system**  
- Evaluate how well your system works  

## Key Concepts

Before starting, let's understand some important terms:

- **Dataset**: A collection of data (here, privacy policies)
- **NLP (Natural Language Processing)**: Techniques to process and analyze text
- **Tokenization**: Splitting text into sentences
- **Preprocessing**: Cleaning text (lowercase, remove symbols)
- **Keywords**: Important words used for detection (e.g., collect, email)
- **Rule-based system**: A system that follows fixed rules instead of learning

## Dataset Used in This Lab

We use a real-world privacy policy dataset from the ACSAC 2020 Voice Privacy Assistant project.

Dataset link:  
https://github.com/CUSecLab/2020-ACSAC-VPA-Privacy-Policy-Analysis/tree/main/dataset

---

## Dataset Description

The dataset contains sentences from real privacy policies.

Each sentence includes:

- **Data types** (email, location, contacts)  
- **Data actions** (collect, share, store)  
- **Purposes** (advertising, analytics, personalization)  
- **Third-party recipients** (advertisers, analytics providers)  

---

## Important Note

<span style="color:red">Recipients refer only to third-party entities</span>

Data usage by the app itself is **not explicitly labeled** in this dataset.

## What is a Data Practice Sentence?

A **data practice sentence** describes how user data is handled.

It may include:

- What data is collected (email, location)
- What action is performed (collect, share, store)
- Who receives the data (third parties)
- The purpose (ads, analytics)

---

### Simple Rule

A sentence is a data practice sentence if it contains:

**Action + (Data OR Purpose OR Recipient)**

---

### ✅ Example

"We collect your email address and share it with analytics providers."

- Data → email  
- Actions → collect, share  
- Recipient → analytics providers  

👉 This is a data practice sentence.

---

### ❌ Not a Data Practice Sentence

"This privacy policy may be updated from time to time."

👉 No data, no action → NOT a data practice sentence.

## Step 1: Load Python Libraries

- pandas → for handling data  
- re → for text cleaning using regular expressions  
- sklearn → for evaluation metrics  


In [ ]:
import pandas as pd
import re
from sklearn.metrics import classification_report, confusion_matrix


## Step 2: Load the ACSAC Privacy Policy Dataset

In this step, we download and extract the privacy policy dataset.

In [ ]:
import requests
import zipfile
import io
import pandas as pd

zip_url = "https://raw.githubusercontent.com/CUSecLab/2020-ACSAC-VPA-Privacy-Policy-Analysis/main/dataset/6_actions_privacy_policy_content.zip"

response = requests.get(zip_url)

zip_file = zipfile.ZipFile(io.BytesIO(response.content))
zip_file.namelist()


['privacy_policy/',
 'privacy_policy/Joke about babies.txt',
 'privacy_policy/paisabazaar.txt',
 'privacy_policy/Robonect lawn mower.txt',
 'privacy_policy/International world football quiz.txt',
 'privacy_policy/World Traveller.txt',
 'privacy_policy/Hoover Dish.txt',
 'privacy_policy/CR Trivia.txt',
 'privacy_policy/Renters Insurance.pdf.txt',
 'privacy_policy/knowledge about sportperson.txt',
 'privacy_policy/Beauty Mirror.txt',
 'privacy_policy/Which Language.txt',
 'privacy_policy/Reliance Fresh.txt',
 'privacy_policy/The Belfast Giants.txt',
 'privacy_policy/LJ Hooker Concierge.txt',
 'privacy_policy/Schwab.txt',
 'privacy_policy/Colors and Style.txt',
 'privacy_policy/Virat Kohli Question.txt',
 'privacy_policy/Jalandhar City Guide.txt',
 'privacy_policy/iFood.txt',
 'privacy_policy/My Sachse.txt',
 'privacy_policy/HSS.txt',
 "privacy_policy/Mahatma Gandhi's Life.txt",
 'privacy_policy/Huntspoint BBQ.txt',
 'privacy_policy/Horo guide.txt',
 'privacy_policy/Parker Family Dental.t

## Step 3: Extract Privacy Policy Files

In this step, we filter and keep only the text files that contain privacy policies.

In [ ]:
import requests
import zipfile
import io
import pandas as pd

# Step 1: Download the ZIP that contains privacy policy content
zip_url = "https://raw.githubusercontent.com/CUSecLab/2020-ACSAC-VPA-Privacy-Policy-Analysis/main/dataset/6_actions_privacy_policy_content.zip"

response = requests.get(zip_url)
z = zipfile.ZipFile(io.BytesIO(response.content))

# List all files inside the ZIP (you already saw this, just keeping it here)
all_files = z.namelist()

# Filter only the privacy policy text files
policy_files = [
    name for name in all_files
    if name.startswith("privacy_policy/") and name.endswith(".txt")
]

len(policy_files), policy_files[:5]


(1910,
 ['privacy_policy/Joke about babies.txt',
  'privacy_policy/paisabazaar.txt',
  'privacy_policy/Robonect lawn mower.txt',
  'privacy_policy/International world football quiz.txt',
  'privacy_policy/World Traveller.txt'])

## Step 4: Build a DataFrame of Policies

In this step, we read each privacy policy file and store:
- the app name
- the full policy text

We organize this data into a pandas DataFrame for further analysis.

In [ ]:
records = []

# You can use all 1910 files, or limit for speed with policy_files[:300]
for fname in policy_files:
    with z.open(fname) as f:
        text = f.read().decode("utf-8", errors="ignore")

    # extract app name from file path: "privacy_policy/<App Name>.txt"
    app_name = fname.split("/", 1)[1].replace(".txt", "")

    records.append({
        "app_name": app_name,
        "policy_text": text
    })

policies_df = pd.DataFrame(records)
policies_df.head()


,app_name,policy_text
0,Joke about babies,_close_\n\nCreating A Voice First World\n\n# ...
1,paisabazaar,* DOWNLOAD APP\n\n![](https://static.paisaba...
2,Robonect lawn mower,# ![](https://robonect.michael-eckel.de/img/lo...
3,International world football quiz,Privacy Policy for International World Footbal...
4,World Traveller,Privacy Policy for World Traveler on the Googl...


## Step 5: Build a Sentence-Level Dataset

### Convert Policy Documents into Sentences

In this step, we split each full privacy policy into individual sentences using `nltk.sent_tokenize`.

Each row in `sentences_df` represents:
- one sentence
- the app it comes from

In [ ]:
import nltk
nltk.download("punkt")
from nltk.tokenize import sent_tokenize

rows = []

for _, row in policies_df.iterrows():
    text = str(row["policy_text"])
    try:
        sentences = sent_tokenize(text)
    except Exception:
        sentences = [text]  # fallback: whole text as one "sentence"

    for sent in sentences:
        sent = sent.strip()
        if len(sent) == 0:
            continue

        rows.append({
            "app_name": row["app_name"],
            "sentence_text": sent
        })

sentences_df = pd.DataFrame(rows)
sentences_df.head()


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


,app_name,sentence_text
0,Joke about babies,_close_\n\nCreating A Voice First World\n\n# P...
1,paisabazaar,* DOWNLOAD APP\n\n![](https://static.paisabaza...
2,Robonect lawn mower,# ![](https://robonect.michael-eckel.de/img/lo...
3,International world football quiz,Privacy Policy for International World Footbal...
4,World Traveller,Privacy Policy for World Traveler on the Googl...


## Step 6: Clean the Sentences (NLP Preprocessing)

Before applying keyword-based detection, we preprocess each sentence to make the text cleaner and more consistent for analysis.

### Preprocessing Steps:
- Convert text to lowercase (case-insensitive matching)
- Remove punctuation, numbers, and special characters
- Normalize whitespace (remove extra spaces)

This step improves the reliability of keyword matching and reduces noise in the text.

💡 This process is called **text preprocessing** in NLP.

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

sentences_df["clean_sentence"] = sentences_df["sentence_text"].apply(clean_text)
sentences_df[["sentence_text", "clean_sentence"]].head()


,sentence_text,clean_sentence
0,_close_\n\nCreating A Voice First World\n\n# P...,close creating a voice first world privacy pol...
1,* DOWNLOAD APP\n\n![](https://static.paisabaza...,download app https static paisabazaar com comp...
2,# ![](https://robonect.michael-eckel.de/img/lo...,https robonect michael eckel de img logo png g...
3,Privacy Policy for International World Footbal...,privacy policy for international world footbal...
4,Privacy Policy for World Traveler on the Googl...,privacy policy for world traveler on the googl...


## Step 7: Define Keyword Dictionaries

In this step, we define lists of keywords that help us identify data practice sentences.

These keywords are grouped into categories:

- **Data types** → What kind of user data is mentioned (e.g., email, location)
- **Actions** → What is done with the data (e.g., collect, share, store)
- **Recipients** → Who receives the data (e.g., advertisers, third parties)
- **Purposes** → Why the data is used (e.g., advertising, analytics)

These keywords will be used to build a rule-based detection system.

In [ ]:
data_type_keywords = [
    "email", "email address", "location", "geolocation",
    "contact", "contacts", "name", "phone", "phone number",
    "payment", "credit card", "ip address", "device id", "account",
    "personal information", "personal data"
]

action_keywords = [
    "collect", "gather", "obtain", "receive",
    "use", "process", "store", "retain",
    "share", "disclose", "sell", "transfer"
]

recipient_keywords = [
    "third party", "third-party", "partners", "affiliates",
    "advertisers", "analytics providers", "service providers",
    "business partners"
]

purpose_keywords = [
    "advertising", "ads", "marketing",
    "analytics", "improve", "personalize", "recommend",
    "research", "statistics", "performance", "provide our services"
]


## Step 8: Detect Data Practice Sentences (Keyword-Based Rules)

In this step, we classify a sentence as a **data practice sentence** using a simple rule.

### Rule:

A sentence is predicted as a data practice sentence if:

1. It contains **at least one action keyword** (e.g., collect, share, use, store), and  
2. It contains **at least one keyword** from any of the following:
   - data type keywords (e.g., email, location)
   - recipient keywords (e.g., third party, advertisers)
   - purpose keywords (e.g., advertising, analytics)

---

### Important Note

A sentence does **not** need all categories.  
Even partial information is enough.

---

### Example

"We collect your email address."

- Action → collect  
- Data → email  

👉 Predicted as Data Practice ✅

In [ ]:
def is_data_practice_sentence(text):
    # text is assumed already lowercased (clean_sentence)
    has_action = any(word in text for word in action_keywords)
    has_other = any(
        word in text
        for word in (data_type_keywords + recipient_keywords + purpose_keywords)
    )
    return has_action and has_other

sentences_df["predicted_is_data_practice"] = sentences_df["clean_sentence"].apply(is_data_practice_sentence)

sentences_df[["app_name", "clean_sentence", "predicted_is_data_practice"]].head(20)


,app_name,clean_sentence,predicted_is_data_practice
0,Joke about babies,close creating a voice first world privacy pol...,True
1,paisabazaar,download app https static paisabazaar com comp...,True
2,Robonect lawn mower,https robonect michael eckel de img logo png g...,True
3,International world football quiz,privacy policy for international world footbal...,True
4,World Traveller,privacy policy for world traveler on the googl...,False
5,Hoover Dish,english italiano deutsch espa ol fran ais esk ...,True
6,CR Trivia,privacy policy for cr trivia on the google ass...,False
7,Renters Insurance.pdf,microsoft word starbutterprivacynotice july do...,True
8,knowledge about sportperson,privacy policy for knowledge about sportperson...,False
9,Beauty Mirror,a text previous next sign in sign up currentla...,True


## Step 9: Explore the Detected Data Practice Sentences

Now that we have a keyword-based detector, let's analyze the results.

### We will:
- Count how many sentences are predicted as data practice
- View example sentences labeled as data practice
- View example sentences labeled as NOT data practice

---

### Observation

Some sentences predicted as *not* data practice are very long and contain multiple ideas.

These sentences may still describe data handling behavior but are missed due to:
- complex sentence structure
- missing keywords
- limitations of preprocessing

This highlights the limitations of simple rule-based approaches.

In [ ]:
# How many sentences in total?
total_sentences = len(sentences_df)
data_practice_sentences = sentences_df["predicted_is_data_practice"].sum()

print(f"Total sentences: {total_sentences}")
print(f"Predicted data practice sentences: {data_practice_sentences}")
print(f"Percentage predicted as data practice: {data_practice_sentences / total_sentences:.2%}")

# Show some examples predicted as data practice
print("\nExample sentences predicted as DATA PRACTICE:\n")
display(
    sentences_df[sentences_df["predicted_is_data_practice"]]
    [["app_name", "sentence_text", "clean_sentence"]]
    .head(10)
)

# Show some examples NOT predicted as data practice
print("\n Example sentences predicted as NOT data practice:\n")
display(
    sentences_df[~sentences_df["predicted_is_data_practice"]]
    [["app_name", "sentence_text", "clean_sentence"]]
    .head(10)
)


Total sentences: 1900
Predicted data practice sentences: 1398
Percentage predicted as data practice: 73.58%

Example sentences predicted as DATA PRACTICE:



,app_name,sentence_text,clean_sentence
0,Joke about babies,_close_\n\nCreating A Voice First World\n\n# P...,close creating a voice first world privacy pol...
1,paisabazaar,* DOWNLOAD APP\n\n![](https://static.paisabaza...,download app https static paisabazaar com comp...
2,Robonect lawn mower,# ![](https://robonect.michael-eckel.de/img/lo...,https robonect michael eckel de img logo png g...
3,International world football quiz,Privacy Policy for International World Footbal...,privacy policy for international world footbal...
5,Hoover Dish,ï»¿ \n \n \n \nEnglish Italiano Deutsch Es...,english italiano deutsch espa ol fran ais esk ...
7,Renters Insurance.pdf,Microsoft Word - StarbutterPrivacyNotice_July2...,microsoft word starbutterprivacynotice july do...
9,Beauty Mirror,{{a.text}}\n\nPrevious Next\n\nSign In / Sign ...,a text previous next sign in sign up currentla...
11,Reliance Fresh,Reliance Fresh Privacy Policy About this poli...,reliance fresh privacy policy about this polic...
12,The Belfast Giants,Ask Belfast Giants\n\n * How To Install\n * ...,ask belfast giants how to install examples con...
13,LJ Hooker Concierge,Toggle navigation\n\n![Property Realm – Offici...,toggle navigation property realm official page...



 Example sentences predicted as NOT data practice:



,app_name,sentence_text,clean_sentence
4,World Traveller,Privacy Policy for World Traveler on the Googl...,privacy policy for world traveler on the googl...
6,CR Trivia,Privacy Policy for CR Trivia on the Google Ass...,privacy policy for cr trivia on the google ass...
8,knowledge about sportperson,Privacy Policy for knowledge about sportperson...,privacy policy for knowledge about sportperson...
10,Which Language,Which Language Privacy Policy Which Language i...,which language privacy policy which language i...
16,Virat Kohli Question,Privacy Policy for Virat Kohli Question on the...,privacy policy for virat kohli question on the...
17,Jalandhar City Guide,Privacy Policy for [Jalandhar city guide] Appl...,privacy policy for jalandhar city guide applic...
24,Parker Family Dental,Privacy Policy for Parker Family Dental on the...,privacy policy for parker family dental on the...
25,bhojpuri songs,Privacy Policy for bhojpuri songs on the Googl...,privacy policy for bhojpuri songs on the googl...
26,Quiz on Indian Premier League,Privacy Policy for Quiz on Indian Premier Leag...,privacy policy for quiz on indian premier leag...
30,Healthy Life,Privacy Policy for Healthy Life on the Google ...,privacy policy for healthy life on the google ...


## Step 10: Create a Small Labeled Set for Evaluation

To evaluate our keyword-based detector, we will:

1. Randomly sample a small set of sentences  
2. Manually label each sentence as:
   - 1 → Data Practice Sentence  
   - 0 → Not a Data Practice Sentence  
3. Compare human labels (ground truth) with model predictions  

This helps us measure how accurate our rule-based system is.

In [ ]:
# Sample 100 sentences for manual labeling (you can change 100 to 50 if needed)
sample_size = 20
sample_df = sentences_df.sample(sample_size, random_state=42).reset_index(drop=True)

# Add a placeholder column for true labels
sample_df["true_is_data_practice"] = None  # students will fill this (0 or 1)

# Show the sample to students
sample_df[["app_name", "clean_sentence", "predicted_is_data_practice", "true_is_data_practice"]].head(10)


,app_name,clean_sentence,predicted_is_data_practice,true_is_data_practice
0,River Ridge Mall,privacy we have created this privacy statement...,True,None
1,Northern Credit Union.pdf,microsoft word ncuprivacypolicy facts what doe...,True,None
2,Bamboo Airways,search this site privacy policy for bamboo air...,False,None
3,cricket demo,privacy policy for cricket demo on the google ...,False,None
4,HomiSmart,smart home pricing smart boiler wireless smart...,True,None
5,Insights with Microsoft,microsoft https img prod cms rt microsoft com ...,True,None
6,Recruiter Lisa,static assets img param logo black header svg ...,True,None
7,Visa Services,travel visa services images travel visa servic...,True,None
8,park detail,search this site https lh googleusercontent co...,True,None
9,Heatmiser Neo,twitter facebook youtube call us shopping cart...,True,None


## Step 11: Manual Annotation by Students

In this step, you will manually label whether each sentence is a data practice sentence.

### Instructions:

- Enter **1** → if the sentence describes data collection, use, sharing, storage, recipients, or purpose  
- Enter **0** → if it does NOT describe a data practice  

You will input labels directly in the next code cell.

---

💡 Note:  
Manual annotation is time-consuming. This highlights the need for automated and machine learning approaches, which we will explore in later labs.

In [ ]:
# Students will label each sentence one-by-one here

for i in range(len(sample_df)):
    print("\n--------------------------------------")
    print(f"Sentence {i+1} / {len(sample_df)}")
    print("App Name:", sample_df.loc[i, "app_name"])
    print("Sentence:", sample_df.loc[i, "clean_sentence"])
    print("Model Prediction:", sample_df.loc[i, "predicted_is_data_practice"])

    label = input("Enter true label (1 = Data Practice, 0 = Not Data Practice): ")

    # Keep asking until valid input is given
    while label not in ["0", "1"]:
        label = input("Invalid input! Enter only 0 or 1: ")

    sample_df.loc[i, "true_is_data_practice"] = int(label)

print("\nManual labeling completed!")



--------------------------------------
Sentence 1 / 20
App Name: River Ridge Mall
Sentence: privacy we have created this privacy statement in order to demonstrate our firm commitment to privacy we are also committed to providing you with the best experience we can on our web site and river ridge mall voice application integration in order to enhance your experience we gather certain information to help us customize our content to your tastes and preferences please read the following policy to understand our information gathering and dissemination practices for this web site and river ridge mall voice application integration the policy may change from time to time so please check back periodically our site uses entry forms when we run contests which require users to give us contact information like their name and email address we use customer contact information from the registration form application and contest entries to send the user information about our company and promotional mat

## Step 12: Save Labeled Data

After manual annotation, we save the labeled dataset for future use.

This allows us to reuse the labeled data without repeating the manual process.

In [ ]:
sample_df.to_csv("lab1_manual_labels.csv", index=False)
print(" Saved as lab1_manual_labels.csv")


 Saved as lab1_manual_labels.csv


## Step 13: Evaluate the Model

In this step, we compare:

- **True labels (human annotations)**  
- **Predicted labels (rule-based system)**  

We use:

- **Classification Report** → precision, recall, F1-score  
- **Confusion Matrix** → correct vs incorrect predictions  

💡 This helps us measure how well our system performs.

### What do these metrics mean?

- **Precision** → How many predicted data practice sentences are actually correct  
- **Recall** → How many real data practice sentences we successfully detected  
- **F1-score** → Balance between precision and recall  

A perfect model would have scores close to 1.0.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Use only rows where student has actually filled a label
mask = sample_df["true_is_data_practice"].notna()

y_true = sample_df.loc[mask, "true_is_data_practice"].astype(int)
y_pred = sample_df.loc[mask, "predicted_is_data_practice"].astype(int)

print("Classification Report:\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Not Data Practice", "Data Practice"]
))

print("Confusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))


Classification Report:

                   precision    recall  f1-score   support

Not Data Practice       0.50      0.22      0.31         9
    Data Practice       0.56      0.82      0.67        11

         accuracy                           0.55        20
        macro avg       0.53      0.52      0.49        20
     weighted avg       0.53      0.55      0.51        20

Confusion Matrix:

[[2 7]
 [2 9]]


## Conclusion

In this lab, we explored how keyword-based NLP techniques can be used to detect data practice sentences in privacy policies.

We learned that:

- Rule-based approaches are simple and easy to interpret  
- However, real-world privacy policies are complex  
- Many sentences contain implicit or unclear descriptions of data usage  

Because of these limitations, keyword-based methods may miss important information.

---

### Future Work

In the next lab, we will build machine learning models that can automatically learn patterns in data practice sentences and improve detection accuracy.

In the next lab, we will build machine learning–based models to automatically learn data practice patterns beyond simple keywords.
